# Lab 2 — Data Cleaning and Preprocessing

**DS331 · Introduction to Data Science**

Real-world data is messy: values are missing, types are wrong, rows are duplicated, and some numbers are simply impossible. Data scientists spend a large share of their time cleaning — and the quality of every later step (statistics, plots, conclusions) depends on it. **Garbage in, garbage out.**

In this notebook you will learn how to:

1. [Audit a dataset for problems](#1)
2. [Handle missing values](#2)
3. [Remove duplicates](#3)
4. [Fix data types](#4)
5. [Clean text columns and rename things](#5)
6. [Detect and handle outliers](#6)
7. [Save the cleaned dataset](#7)
8. [Try it yourself ✏️](#8)

**Dataset:** the *Titanic* passenger list — 891 passengers, with age, ticket fare, class, and whether they survived. A classic, and genuinely messy: about 20% of ages are missing.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

titanic = sns.load_dataset("titanic")
titanic.head()

<a id="1"></a>
## 1. Auditing: find the problems first

Cleaning starts with a diagnosis, not with deleting things. Run the standard checks from Lab 1:

In [ ]:
titanic.info()

Spotted from `info()` alone:
- `age` has 714 non-null values out of 891 → **177 missing**.
- `deck` has only 203 non-null values → **77% missing**!
- `embarked` / `embark_town` are missing 2 values.

### Counting missing values explicitly

`isna()` returns True/False for every cell; summing counts the Trues (True counts as 1).

In [ ]:
missing = titanic.isna().sum()
missing_pct = (titanic.isna().mean() * 100).round(1)

pd.DataFrame({"missing_count": missing, "missing_%": missing_pct}).sort_values(
    "missing_count", ascending=False
)

### Visualizing missingness

A heatmap of `isna()` shows *where* the holes are — each bright line is a missing value. Patterns matter: scattered holes are easier to handle than entire missing blocks.

In [ ]:
plt.figure(figsize=(10, 5))
sns.heatmap(titanic.isna(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing values per column (bright = missing)")
plt.show()

<a id="2"></a>
## 2. Handling missing values

There are two families of strategies: **drop** the missing data, or **fill** it (called *imputation*). Which is right depends on *how much* is missing and *why*.

| Situation | Sensible strategy |
|---|---|
| A column is mostly missing (like `deck`, 77%) | drop the **column** |
| A few rows are missing in an important column | drop those **rows** |
| A moderate share missing in a numeric column | fill with **median** (or mean) |
| A moderate share missing in a categorical column | fill with the **mode** or a label like `"Unknown"` |

We always work on a **copy**, so the raw data stays untouched and we can re-run cells safely:

In [ ]:
df = titanic.copy()

### 2.1 Dropping

`dropna()` removes rows (default) or columns (`axis=1`) containing missing values.

> ⚠️ **Common mistake:** calling `df.dropna()` with no arguments drops every row that has *any* missing value — here that would silently delete most of the dataset. Always check the shape before and after.

In [ ]:
print("if we dropped every row with any NaN:", titanic.dropna().shape, "<- only ~180 rows left!")

# Drop the 'deck' column: 77% missing is too much to fill credibly
df = df.drop(columns=["deck"])

# Drop the 2 rows where 'embarked' is missing: a tiny, safe loss
df = df.dropna(subset=["embarked"])

print("our shape after targeted cleaning:  ", df.shape)

### 2.2 Filling (imputation)

`age` is missing for ~20% of passengers — too many rows to throw away. We fill instead.

For a numeric column the **median** is usually safer than the mean because it is not pulled around by outliers.

In [ ]:
print("age before filling -> missing:", df["age"].isna().sum(),
      "| median:", df["age"].median())

df["age"] = df["age"].fillna(df["age"].median())

print("age after filling  -> missing:", df["age"].isna().sum())

Other `fillna` variants you will meet:

```python
df["col"].fillna(0)                            # a constant
df["col"].fillna(df["col"].mean())             # the mean
df["col"].fillna(df["col"].mode()[0])          # most frequent value (note the [0])
df["col"].ffill()                              # copy the previous value down (time series)
```

> ⚠️ **Be honest in your report.** Imputation invents data. Always *state* what you filled and how ("age: 177 missing values filled with the median, 28.0") — never hide it.

In [ ]:
# Final check: no missing values left
df.isna().sum().sum()

<a id="3"></a>
## 3. Duplicates

Duplicate rows usually come from data-entry or merging mistakes, and they bias every count and average.

In [ ]:
print("duplicated rows:", df.duplicated().sum())

In [ ]:
# Look at a few of them before deciding anything
df[df.duplicated(keep=False)].sort_values(["age", "fare"]).head(6)

Here the "duplicates" are passengers who happen to share every recorded attribute — the dataset has no name/ID column to tell them apart. In the *original* Titanic data (with names) these are different people, so we **keep** them.

That is the lesson: **inspect duplicates before dropping them.** When they really are repeated records, this removes them:

```python
df = df.drop_duplicates()                    # exact duplicate rows
df = df.drop_duplicates(subset=["user_id"])  # duplicates by a key column
```

<a id="4"></a>
## 4. Fixing data types

Wrong types cause subtle bugs: you cannot compute the mean of a number stored as text, and a date stored as text cannot be sorted chronologically.

In [ ]:
df.dtypes

### 4.1 `astype()` — direct conversion

`survived` and `pclass` are stored as integers, but they are really **categories** (you would never compute "average passenger class = 2.31" and present it as meaningful). The `category` dtype documents this and saves memory.

In [ ]:
df["survived"] = df["survived"].astype("category")
df["pclass"] = df["pclass"].astype("category")
df[["survived", "pclass"]].dtypes

### 4.2 Numbers trapped in text — `pd.to_numeric`

A very common situation with messy CSVs: a numeric column arrives as `object` because some entries contain junk. A small fabricated example:

In [ ]:
prices = pd.Series(["12.5", "8.99", "n/a", "15.0", "?"])
print(prices.dtype)                      # object = text!

clean_prices = pd.to_numeric(prices, errors="coerce")   # junk becomes NaN
clean_prices

`errors="coerce"` converts anything unparseable to `NaN` — which you then handle with the Section 2 toolbox. Sometimes you must strip symbols first:

```python
df["price"] = df["price"].str.replace("$", "", regex=False).str.replace(",", "")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
```

### 4.3 Dates — `pd.to_datetime`

Dates stored as text become powerful once converted (we exploit this fully in Lab 3):

In [ ]:
dates = pd.Series(["2024-01-15", "2024-02-03", "2024-03-21"])
dates = pd.to_datetime(dates)
print(dates.dtype)
dates.dt.day_name()

<a id="5"></a>
## 5. Cleaning text and renaming

### 5.1 The `.str` accessor

Text columns often hide inconsistencies — extra spaces, mixed capitalization — which make `"Yes"`, `"yes "` and `"YES"` count as three different values.

In [ ]:
messy_text = pd.Series(["  Male", "female", "MALE  ", " Female", "male"])

cleaned = messy_text.str.strip().str.lower()
print(cleaned.tolist())
print(cleaned.value_counts())

The pattern: `.str.strip()` removes surrounding whitespace, `.str.lower()` unifies the case. Also useful: `.str.replace(...)`, `.str.contains(...)`, `.str.split(...)`.

### 5.2 Renaming columns

Clear, consistent column names (lowercase, underscores, no spaces) make every later line of code easier to write.

In [ ]:
df = df.rename(columns={"sibsp": "siblings_spouses", "parch": "parents_children"})
list(df.columns)

To clean *all* names at once (very handy for datasets with names like `" Total Price ($) "`):

```python
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
```

<a id="6"></a>
## 6. Outliers

An **outlier** is a value far away from the rest. Two very different kinds:

- **Errors** — a human age of 999, a negative price. Fix or remove.
- **Real but extreme** — a genuinely expensive ticket. Usually **keep** (it is information!), but be aware it can dominate means and stretch plots.

### 6.1 Spotting outliers

`describe()` and a **boxplot** are the standard tools. In a boxplot, the box spans the middle 50% of the data and the dots beyond the whiskers are potential outliers.

In [ ]:
df["fare"].describe()

In [ ]:
plt.figure(figsize=(8, 2.5))
sns.boxplot(x=df["fare"])
plt.title("Ticket fares — a long tail of expensive tickets")
plt.show()

The median fare is ~14, yet the maximum is 512 — a few passengers paid enormous fares.

### 6.2 The IQR rule — a standard definition

The most common rule of thumb flags values more than 1.5 × IQR beyond the quartiles, where IQR = Q3 − Q1 (exactly the boxplot's whisker rule):

In [ ]:
q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df[(df["fare"] < lower) | (df["fare"] > upper)]
print(f"IQR bounds: [{lower:.1f}, {upper:.1f}]")
print(f"flagged as outliers: {len(outliers)} rows ({len(outliers)/len(df)*100:.1f}%)")

### 6.3 What to do about them

These expensive fares are **real** (first-class suites), so we keep them — and note the skew for our plots. If you do need to limit their influence, the gentle option is **capping** rather than deleting:

In [ ]:
df["fare_capped"] = df["fare"].clip(upper=upper)

fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
axes[0].hist(df["fare"], bins=40)
axes[0].set_title("fare (original)")
axes[1].hist(df["fare_capped"], bins=40)
axes[1].set_title("fare (capped at IQR upper bound)")
plt.show()

> ⚠️ **Never silently delete outliers** just to make plots prettier. In your report, state the rule you used and how many rows were affected.

<a id="7"></a>
## 7. Saving the cleaned dataset

Save the result so the next notebook (and your future self) can start from clean data. `index=False` avoids writing the row numbers as an extra column.

In [ ]:
df.to_csv("titanic_clean.csv", index=False)

# Read it back to confirm the round trip worked
check = pd.read_csv("titanic_clean.csv")
print("saved and reloaded:", check.shape)
check.head(3)

### The cleaning checklist for your report

For your assigned dataset, walk through exactly what we did here:

1. `info()` + `isna().sum()` — what is missing, what has the wrong type?
2. Decide per column: drop or fill? **Write down your choice and the reason.**
3. Check duplicates — inspect before dropping.
4. Fix dtypes (numbers, categories, dates).
5. Clean text columns; standardize column names.
6. Look for outliers; decide keep / cap / remove — and say so.
7. Save the cleaned file and use it for everything that follows.

<a id="8"></a>
## 8. Try it yourself ✏️

Below is a small, deliberately messy dataset. Apply the full checklist to it.

In [ ]:
messy = pd.DataFrame({
    "Name ": ["  alice", "Bob", "bob", "CHARLIE ", "dana", "Bob", None],
    "Age": ["25", "32", "32", "199", "28", "32", "41"],
    "City": ["Dhaka", " dhaka", "Chittagong", "Dhaka", None, " dhaka", "Sylhet"],
    "Salary($)": ["50,000", "62,000", "62,000", "58,000", "n/a", "62,000", "71,000"],
})
messy

**Exercise 1.** Standardize the column names: strip spaces, lowercase, and replace `($)` so the names are valid identifiers like `salary`.

**Exercise 2.** `Age` is stored as text and contains an impossible value. Convert it to numeric and deal with the 199. Justify your choice.

**Exercise 3.** Clean the `Name` and `City` text (strip, consistent case), then find and remove the truly duplicated row.

**Exercise 4.** Convert `Salary($)` to numeric (careful: commas and `"n/a"`), then fill the missing salary with a value you think is reasonable — and write one sentence defending the choice.

**Exercise 5.** Save your cleaned DataFrame to `messy_clean.csv` and reload it to verify.

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4

In [ ]:
# Exercise 5

---
**Next lab:** clean data is the foundation — now we *create new columns* that make analysis sharper: **Lab 3 — Feature Engineering**.